# Location Named Entity Recognition (NER) - Training Pipeline (Google Colab Pro)
### Bi-LSTM / Bi-GRU Architecture with Dynamic Threshold Tuning for Ukrainian/Russian Datasets

**Competition Constraints Strictly Followed:**
- **PyTorch Only**: No Keras, TensorFlow, or Chainer.
- **Recurrent Only**: Bi-LSTM / Bi-GRU architectures with embeddings and linear projections (No Transformers or BERT).
- **Direct Metric Optimization**: Evaluated on exact set intersections via `hard_processing` using the official `comp_metric`.
- **Threshold Optimization**: Sweeping probability cutoff from 0.3 to 0.8 to maximize set-level F1.


In [ ]:
# Cell 1: Colab Setup & Drive Mount
from google.colab import drive
drive.mount('/content/drive')

# Install modern PyTorch Lightning and Weights & Biases
!pip install -q pytorch-lightning wandb


In [ ]:
# Cell 2: Extract Data & Sync Project Modules
import os
import shutil

# Copy competition zip archive from Google Drive to local /content/ SSD
!cp /content/drive/MyDrive/DL/Assignment-3/kse-ua-location-extraction-2026.zip /content/
!unzip -q -o /content/kse-ua-location-extraction-2026.zip -d /content/

# Sync src/ package from Drive if running directly from Colab root
drive_src = "/content/drive/MyDrive/DL/Assignment-3/src"
local_src = "/content/src"

if os.path.exists(drive_src) and not os.path.exists(local_src):
    shutil.copytree(drive_src, local_src)
    print(f"Successfully copied 'src' package from {drive_src} to {local_src}")
elif os.path.exists(local_src):
    print(f"Using local 'src' package at {local_src}")
else:
    print("Warning: Ensure 'src/' folder is present in /content/ or Google Drive folder.")

# List extracted files in /content
print("Extracted content directory contents:")
!ls -lh /content


In [ ]:
# Cell 3: Imports & Reproducibility Setup
import sys
import os
from typing import List, Dict, Any, Tuple
import ast
from datetime import datetime

# Ensure /content is on python path
if '/content' not in sys.path:
    sys.path.append('/content')
drive_root = '/content/drive/MyDrive/DL/Assignment-3'
if drive_root not in sys.path and os.path.exists(drive_root):
    sys.path.append(drive_root)

import torch
import torch.nn as nn
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold

# Import our custom modules
from src.preprocessing import (
    hard_processing,
    comp_metric,
    comp_metric_detailed,
    clean_entity_boundaries,
    WordTokenizer,
    WordVocab,
    CharVocab,
    extract_entities_from_bio,
    extract_entities_from_probs,
    parse_loc_markers,
    parse_locations_list,
)
from src.dataset import (
    NERDataset,
    NERCollate,
    load_dataset_csv,
    build_vocab_from_texts,
    create_dataloaders,
)
from src.models import (
    RecurrentNERLightningModule,
    ThresholdOptimizationCallback,
)

# Seed for strict reproducibility
SEED = 42
pl.seed_everything(SEED, workers=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using execution device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 4: Experiment Configuration
class Config:
    # Paths
    DATA_DIR = '/content'
    DRIVE_DIR = '/content/drive/MyDrive/DL/Assignment-3'
    CHECKPOINT_DIR = '/content/drive/MyDrive/DL/Assignment-3/checkpoints'
    SUBMISSION_DIR = '/content/drive/MyDrive/DL/Assignment-3'
    
    # Dataset files
    UK_DATA_FILE = 'uk_geo_dataset.csv'
    RU_DATA_FILE = 'ru_geo_dataset.csv'
    LABELING_SAMPLE_FILE = 'labeling_sample.csv'
    TEST_FILE = 'test.csv'
    
    # Data preprocessing
    MAX_LEN = 128            # Covers 99% of sample word lengths without truncation
    VOCAB_SIZE = 60000       # Vocabulary capacity
    MIN_FREQ = 2             # Minimum word frequency for vocabulary
    SCHEME = 'BIO'           # 'BIO' (3 classes) or 'IO' (2 classes)
    NUM_CLASSES = 3 if SCHEME == 'BIO' else 2
    
    # Model architecture (Bi-LSTM with Char-CNN Morphology)
    RNN_TYPE = 'LSTM'        # 'LSTM' or 'GRU'
    EMBEDDING_DIM = 128
    HIDDEN_DIM = 256
    NUM_LAYERS = 2
    BIDIRECTIONAL = True
    DROPOUT = 0.30
    SPATIAL_DROPOUT = 0.20
    
    # Character-level CNN parameters (Morphological subword feature extractor)
    USE_CHAR_CNN = True
    CHAR_EMBED_DIM = 32
    CHAR_FEATURE_DIM = 64
    MAX_CHAR_LEN = 16
    
    # Training & Loss
    BATCH_SIZE = 64
    VAL_BATCH_SIZE = 128
    LR = 1e-3
    WEIGHT_DECAY = 1e-5
    MAX_EPOCHS = 8
    # Calibrated class weights: 15.0 achieves ideal balance between precision & recall
    CLASS_WEIGHTS = [1.0, 15.0, 15.0] if SCHEME == 'BIO' else [1.0, 15.0]
    
    # Cross Validation & Subsampling
    N_SPLITS = 5
    MAX_TRAIN_SAMPLES = 400000 
    
    # W&B Logging
    WANDB_PROJECT = 'kse-ua-location-extraction-2026'
    WANDB_RUN_NAME = f'bilstm_charcnn_h{HIDDEN_DIM}_l{NUM_LAYERS}_cw15_400k'

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {Config.CHECKPOINT_DIR}')


In [ ]:
# Cell 5: Data Loading & Vocabulary Building (Word + Character)
print("Loading datasets...")

# Load test set (477 evaluation samples)
df_test = load_dataset_csv(Config.TEST_FILE, data_dir=Config.DATA_DIR)
print(f"Loaded test.csv: {len(df_test)} samples")

# Load labeling sample (annotated validation benchmark)
df_sample = load_dataset_csv(Config.LABELING_SAMPLE_FILE, data_dir=Config.DATA_DIR)
print(f"Loaded labeling_sample.csv: {len(df_sample)} samples")

# Load Ukrainian dataset
df_uk = load_dataset_csv(
    Config.UK_DATA_FILE,
    data_dir=Config.DATA_DIR,
    nrows=Config.MAX_TRAIN_SAMPLES,
    filter_empty=False
)
print(f"Loaded uk_geo_dataset.csv: {len(df_uk)} samples")

# 1. Build Word-level vocabulary
vocab_path = os.path.join(Config.CHECKPOINT_DIR, "word_vocab.json")
tokenizer = WordTokenizer(lowercase=False)

if os.path.exists(vocab_path):
    print(f"Loading existing vocabulary from {vocab_path}...")
    vocab = WordVocab.load(vocab_path)
else:
    print("Building vocabulary from corpus...")
    vocab = build_vocab_from_texts(
        texts=df_uk['text'].tolist(),
        tokenizer=tokenizer,
        min_freq=Config.MIN_FREQ,
        max_size=Config.VOCAB_SIZE,
        save_path=vocab_path
    )
print(f"Word vocabulary successfully built! Total words: {len(vocab):,}")

# 2. Build Character-level vocabulary for Char-CNN
char_vocab = CharVocab()
char_vocab_path = os.path.join(Config.CHECKPOINT_DIR, "char_vocab.json")
char_vocab.save(char_vocab_path)
print(f"Char vocabulary initialized and saved! Total characters: {len(char_vocab)}")


In [ ]:
# Cell 6: Robust Stratified Cross-Validation Setup
print("Setting up Stratified Cross-Validation...")

# Create stratification label: whether sample contains at least one location entity
def has_location_entity(markers_entry):
    if pd.isna(markers_entry):
        return 0
    s = str(markers_entry).strip()
    return 1 if (s and s != '[]') else 0

df_uk['has_loc'] = df_uk['loc_markers'].apply(has_location_entity)
loc_pct = df_uk['has_loc'].mean() * 100
print(f"Positive location rate across training samples: {loc_pct:.2f}%")

# Stratified K-Fold ensures identical entity balance across all folds
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=SEED)
df_uk['fold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_uk, df_uk['has_loc'])):
    df_uk.loc[val_idx, 'fold'] = fold_idx

fold_counts = df_uk.groupby(['fold', 'has_loc']).size().unstack()
print("Fold distribution:")
display(fold_counts)


In [ ]:
# Cell 7: Model Training Loop with Char-CNN (Fold 0)
ACTIVE_FOLD = 0

train_df = df_uk[df_uk['fold'] != ACTIVE_FOLD].reset_index(drop=True)
val_df = df_uk[df_uk['fold'] == ACTIVE_FOLD].reset_index(drop=True)

print(f"Training on Fold {ACTIVE_FOLD}: {len(train_df):,} train samples, {len(val_df):,} val samples")

# Build DataLoaders with CharVocab support
train_loader, val_loader, test_loader, _ = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    test_df=df_test,
    vocab=vocab,
    char_vocab=char_vocab,
    tokenizer=tokenizer,
    batch_size=Config.BATCH_SIZE,
    val_batch_size=Config.VAL_BATCH_SIZE,
    max_len=Config.MAX_LEN,
    max_char_len=Config.MAX_CHAR_LEN,
    scheme=Config.SCHEME,
    num_workers=2
)

# Initialize Bi-LSTM + Char-CNN PyTorch Lightning Module
model = RecurrentNERLightningModule(
    vocab_size=len(vocab),
    embedding_dim=Config.EMBEDDING_DIM,
    hidden_dim=Config.HIDDEN_DIM,
    num_layers=Config.NUM_LAYERS,
    num_classes=Config.NUM_CLASSES,
    rnn_type=Config.RNN_TYPE,
    bidirectional=Config.BIDIRECTIONAL,
    dropout=Config.DROPOUT,
    spatial_dropout=Config.SPATIAL_DROPOUT,
    lr=Config.LR,
    weight_decay=Config.WEIGHT_DECAY,
    scheme=Config.SCHEME,
    class_weights=Config.CLASS_WEIGHTS,
    scheduler_type='cosine',
    max_epochs=Config.MAX_EPOCHS,
    use_char_cnn=Config.USE_CHAR_CNN,
    char_vocab_size=len(char_vocab),
    char_embed_dim=Config.CHAR_EMBED_DIM,
    char_feature_dim=Config.CHAR_FEATURE_DIM
)

# Setup W&B Logger
wandb_logger = WandbLogger(
    project=Config.WANDB_PROJECT,
    name=f"{Config.WANDB_RUN_NAME}_fold{ACTIVE_FOLD}",
    log_model=False
)
wandb_logger.watch(model.model, log='all', log_freq=500)

# Checkpointing: Save model with highest validation set-level F1 (comp_metric)
checkpoint_callback = ModelCheckpoint(
    dirpath=Config.CHECKPOINT_DIR,
    filename=f"bilstm_charcnn_fold_{ACTIVE_FOLD}_" + "{epoch:02d}_{val_comp_f1:.4f}",
    monitor="val_comp_f1",
    mode="max",
    save_top_k=1,
    save_last=True
)

early_stopping_callback = EarlyStopping(
    monitor="val_comp_f1",
    patience=3,
    mode="max"
)

threshold_callback = ThresholdOptimizationCallback()
lr_monitor = LearningRateMonitor(logging_interval='step')

# Setup PyTorch Lightning Trainer
trainer = pl.Trainer(
    max_epochs=Config.MAX_EPOCHS,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    logger=wandb_logger,
    callbacks=[checkpoint_callback, early_stopping_callback, threshold_callback, lr_monitor],
    precision='16-mixed' if torch.cuda.is_available() else 32,
    gradient_clip_val=1.0,
    log_every_n_steps=50
)

print("Starting training with Char-CNN + Bi-LSTM...")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print(f"Training completed! Best checkpoint saved at: {checkpoint_callback.best_model_path}")
print(f"Best validation comp_f1 achieved: {checkpoint_callback.best_model_score:.6f}")


In [ ]:
# Cell 8: Validation Evaluation on Hand-Annotated Labeling Sample
print("Loading best checkpoint for evaluation...")
best_model = RecurrentNERLightningModule.load_from_checkpoint(
    checkpoint_callback.best_model_path
)
best_model.eval()
best_model.to(device)

# Create DataLoader for labeling_sample.csv with Char-CNN support
_, benchmark_loader, _, _ = create_dataloaders(
    train_df=df_sample,
    val_df=df_sample,
    vocab=vocab,
    char_vocab=char_vocab,
    tokenizer=tokenizer,
    batch_size=Config.VAL_BATCH_SIZE,
    max_len=Config.MAX_LEN,
    max_char_len=Config.MAX_CHAR_LEN,
    scheme=Config.SCHEME
)

print("Evaluating best model on labeling_sample benchmark...")
bench_trainer = pl.Trainer(accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1, logger=False)
bench_results = bench_trainer.validate(best_model, dataloaders=benchmark_loader)

print(f"Benchmark Results on labeling_sample.csv: {bench_results}")


In [ ]:
# Cell 9: Test Inference & Protected Submission File Generation
import glob
from datetime import datetime

print("Generating predictions on test.csv (477 samples)...")

# Predict using best model
predictions = trainer.predict(best_model, dataloaders=test_loader)

all_text_ids = []
all_pred_locations = []

for batch_pred in predictions:
    all_text_ids.extend(batch_pred['text_ids'])
    for loc_list in batch_pred['predicted_locations']:
        cleaned_locs = []
        for loc in loc_list:
            cleaned = clean_entity_boundaries(loc)
            if cleaned and hard_processing(cleaned):
                cleaned_locs.append(cleaned)
        # Deduplicate while preserving sequence order
        deduped = list(dict.fromkeys(cleaned_locs))
        all_pred_locations.append(deduped)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'text_id': all_text_ids,
    'locations': [str(locs) for locs in all_pred_locations]
})

# Verify submission format matches test.csv
assert len(submission_df) == 477, f"Row count mismatch: {len(submission_df)} != 477"

# Format versioned filename with timestamp and score to PRESERVE previous submissions
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
best_score = float(checkpoint_callback.best_model_score) if checkpoint_callback.best_model_score is not None else 0.0
score_str = f"{best_score:.4f}".replace(".", "p")

# 1. Save uniquely versioned file (NEVER overwrites old submissions)
versioned_filename = f"submission_{Config.WANDB_RUN_NAME}_val{score_str}_{timestamp}.csv"
versioned_path = os.path.join(Config.SUBMISSION_DIR, versioned_filename)
submission_df.to_csv(versioned_path, index=False)
print(f"\n[PROTECTED] Versioned submission saved to: {versioned_path}")

# 2. Also save to submission_latest.csv for quick reference
latest_path = os.path.join(Config.SUBMISSION_DIR, "submission_latest.csv")
submission_df.to_csv(latest_path, index=False)
print(f"[CONVENIENCE] Latest submission copy saved to: {latest_path}")

print(f"\nTotal rows in submission: {len(submission_df)} (Verified 477)")
print("Sample submission predictions:")
display(submission_df.head(10))

# Finish W&B run
wandb.finish()
print("Pipeline complete!")
